# LLM Evaluation Notebook

## 1. Imports & Setup

In [1]:
import torch, gc, time, json, os
from transformers import AutoTokenizer, AutoModelForCausalLM

RESULT_FILE = "llm_test_results.json"

# Load existing results or start fresh
if os.path.exists(RESULT_FILE):
    with open(RESULT_FILE, encoding="utf-8") as f:
        all_results = json.load(f)
    print(f"Loaded {len(all_results)} existing results from {RESULT_FILE}")
else:
    all_results = []
    print("Starting fresh results list.")


Starting fresh results list.


## 2. Test Prompts & Inputs

In [ ]:
test_prompts_by_type = {
    # Each prompt is a plain instruction string.
    # user_id injection and input scoping are handled in run_model().
    "classification": [
        "Based on the selected user's UserProfile (height_cm, weight_kg, body_fat_pct, age, activity_level), classify their most suitable fitness goal. Possible labels: fat_loss, muscle_gain, maintain. Return only the label.",
        "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), determine whether their stated goal is appropriate for their current physical condition. Possible labels: match, mismatch. Return only the label.",
        "Based on the selected user's MealLog records, classify their eating habit by examining which meal_type entries (breakfast, lunch, dinner) appear or are missing across dates. Possible labels: regular_three_meals, skip_breakfast, skip_lunch, skip_dinner, irregular_meals. Return only the label.",
        "Based on the selected user's MealLog records, classify their food preference by examining the meal_name entries. Possible labels: asian_only, western_only, salad_heavy, mixed. Return only the label.",
        "Based on the selected user's MealLog records, classify their app engagement level by counting meals logged per day and the number of distinct dates. Possible labels: low_engagement, moderate_engagement, high_engagement. Return only the label.",
    ],
    "prediction": [
        "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), predict their weight trend over the next 30 days. Return a short prediction with expected direction and estimated change.",
        "Based on the selected user's MealLog (calories, protein_g, carbs_g, fat_g per meal) and UserProfile (goal, activity_level), predict whether their current intake pattern will support their goal over the next 30 days. Return a short prediction.",
        "Based on the selected user's MealLog records, predict whether this user is likely to continue logging meals next week, based on the frequency and recency of their entries. Return: likely or unlikely, with a one-line reason.",
        "Based on the selected user's WeightLog trend, BodyFatLog trend, and UserProfile (goal, activity_level), predict whether they are at risk of not achieving their fitness goal. Return: at_risk or on_track, with a one-line reason.",
        "Based on the selected user's MealLog records, predict whether their eating habit is likely to change in the next month, based on patterns in meal_type and meal_name. Return a short prediction.",
    ],
    "structured_extraction": [
        "From the selected user's UserNote, extract structured information about their eating preferences. Return as key-value pairs.",
        "From the selected user's UserNote, extract structured information about their activity level and exercise habits. Return as key-value pairs.",
        "From the selected user's UserNote, extract structured information about their fitness goal, including target amount and timeline if mentioned. Return as key-value pairs.",
        "From the selected user's UserNote, extract any dietary restrictions mentioned. Return as key-value pairs.",
        "From the selected user's UserNote, extract body condition information that may affect their exercise or diet plan. Return as key-value pairs.",
    ],
    "summarization": [
        "Based on the selected user's MealLog records, summarize their overall eating behavior in one sentence, covering meal timing, calorie intake, and nutritional balance.",
        "Based on the selected user's UserProfile (weight_kg, height_cm, body_fat_pct, activity_level, goal), provide a one-line summary of whether their stated goal is realistic given their current body metrics.",
        "Based on the selected user's MealLog records, summarize their diet pattern in one sentence, focusing on meal frequency, meal types, and any noticeable nutritional trends.",
        "Based on the selected user's MealLog records and BodyFatLog history, summarize their level of progress and consistency in one sentence.",
        "Based on the selected user's WeightLog history, BodyFatLog history, and UserProfile (goal, activity_level), summarize their overall progress toward their fitness goal in one sentence.",
    ],
    "text_generation": [
        "Generate a personalised good morning greeting for a user who is on a fat loss goal.",
        "Generate a short motivational message for a user who has been consistently logging meals but has not seen weight change in 2 weeks.",
        "Generate one practical healthy eating tip suitable for a university student with a busy schedule.",
        "Generate a short reminder message for a user who has not logged any meals in the past 3 days.",
        "Generate one follow-up question to better understand why a user keeps skipping breakfast.",
    ],
}

print("Prompts ready. Total task types:", len(test_prompts_by_type))


In [3]:
# -------------------------------------------------------------------------
# Unified user database passed as input to every prompt.
#
# UserProfile:  user_id, sex, age, height_cm, weight_kg, body_fat_pct,
#               activity_level, goal
# WeightLog:    user_id, date, weight_kg
# BodyFatLog:   user_id, date, body_fat_pct  (updated ~monthly per user)
# MealLog:      user_id, date, meal_type, meal_name,
#               calories, protein_g, carbs_g, fat_g
# UserNote:     user_id, note
# -------------------------------------------------------------------------

test_input = """
-- UserProfile --
user_id: 1 | sex: male   | age: 19 | height_cm: 175.00 | weight_kg: 92.00 | body_fat_pct: 28.5 | activity_level: moderate | goal: fat_loss
user_id: 2 | sex: male   | age: 21 | height_cm: 168.00 | weight_kg: 85.00 | body_fat_pct: 22.0 | activity_level: light    | goal: muscle_gain
user_id: 3 | sex: male   | age: 22 | height_cm: 180.00 | weight_kg: 78.00 | body_fat_pct: 18.4 | activity_level: moderate | goal: fat_loss
user_id: 4 | sex: female | age: 25 | height_cm: 162.00 | weight_kg: 58.00 | body_fat_pct: 26.1 | activity_level: moderate | goal: fat_loss
user_id: 5 | sex: female | age: 30 | height_cm: 165.00 | weight_kg: 62.00 | body_fat_pct: 24.8 | activity_level: active   | goal: muscle_gain

-- WeightLog --
user_id: 1 | date: 2026-01-01 | weight_kg: 94.20
user_id: 1 | date: 2026-01-08 | weight_kg: 93.80
user_id: 1 | date: 2026-01-15 | weight_kg: 93.40
user_id: 1 | date: 2026-01-22 | weight_kg: 93.10
user_id: 1 | date: 2026-01-29 | weight_kg: 92.70
user_id: 1 | date: 2026-02-05 | weight_kg: 92.30
user_id: 1 | date: 2026-02-12 | weight_kg: 92.00
user_id: 2 | date: 2026-01-01 | weight_kg: 85.00
user_id: 2 | date: 2026-01-15 | weight_kg: 85.20
user_id: 2 | date: 2026-02-01 | weight_kg: 85.60
user_id: 2 | date: 2026-02-12 | weight_kg: 85.90
user_id: 3 | date: 2026-01-01 | weight_kg: 79.50
user_id: 3 | date: 2026-01-15 | weight_kg: 79.10
user_id: 3 | date: 2026-02-01 | weight_kg: 78.80
user_id: 3 | date: 2026-02-12 | weight_kg: 78.00
user_id: 4 | date: 2026-01-01 | weight_kg: 59.20
user_id: 4 | date: 2026-01-15 | weight_kg: 58.90
user_id: 4 | date: 2026-02-01 | weight_kg: 58.50
user_id: 4 | date: 2026-02-12 | weight_kg: 58.00
user_id: 5 | date: 2026-01-01 | weight_kg: 61.00
user_id: 5 | date: 2026-01-15 | weight_kg: 61.40
user_id: 5 | date: 2026-02-01 | weight_kg: 62.00
user_id: 5 | date: 2026-02-12 | weight_kg: 62.30

-- BodyFatLog --
user_id: 1 | date: 2025-11-01 | body_fat_pct: 30.2
user_id: 1 | date: 2025-12-01 | body_fat_pct: 29.6
user_id: 1 | date: 2026-01-01 | body_fat_pct: 29.1
user_id: 1 | date: 2026-02-01 | body_fat_pct: 28.5
user_id: 2 | date: 2025-11-01 | body_fat_pct: 21.2
user_id: 2 | date: 2025-12-01 | body_fat_pct: 21.5
user_id: 2 | date: 2026-01-01 | body_fat_pct: 21.8
user_id: 2 | date: 2026-02-01 | body_fat_pct: 22.0
user_id: 3 | date: 2025-11-01 | body_fat_pct: 20.1
user_id: 3 | date: 2025-12-01 | body_fat_pct: 19.5
user_id: 3 | date: 2026-01-01 | body_fat_pct: 19.0
user_id: 3 | date: 2026-02-01 | body_fat_pct: 18.4
user_id: 4 | date: 2025-11-01 | body_fat_pct: 28.0
user_id: 4 | date: 2025-12-01 | body_fat_pct: 27.4
user_id: 4 | date: 2026-01-01 | body_fat_pct: 26.8
user_id: 4 | date: 2026-02-01 | body_fat_pct: 26.1
user_id: 5 | date: 2025-11-01 | body_fat_pct: 25.6
user_id: 5 | date: 2025-12-01 | body_fat_pct: 25.3
user_id: 5 | date: 2026-01-01 | body_fat_pct: 25.0
user_id: 5 | date: 2026-02-01 | body_fat_pct: 24.8

-- MealLog --
user_id: 1 | date: 2026-02-10 | meal_type: lunch     | meal_name: chicken salad         | calories: 420 | protein_g: 38 | carbs_g:  18 | fat_g: 14
user_id: 1 | date: 2026-02-10 | meal_type: dinner    | meal_name: steak with rice       | calories: 780 | protein_g: 52 | carbs_g:  65 | fat_g: 28
user_id: 1 | date: 2026-02-11 | meal_type: lunch     | meal_name: tuna sandwich         | calories: 510 | protein_g: 34 | carbs_g:  48 | fat_g: 16
user_id: 1 | date: 2026-02-11 | meal_type: dinner    | meal_name: pasta                 | calories: 690 | protein_g: 22 | carbs_g: 110 | fat_g: 18
user_id: 1 | date: 2026-02-12 | meal_type: lunch     | meal_name: chicken wrap          | calories: 560 | protein_g: 40 | carbs_g:  52 | fat_g: 17
user_id: 1 | date: 2026-02-12 | meal_type: dinner    | meal_name: grilled salmon        | calories: 620 | protein_g: 55 | carbs_g:  10 | fat_g: 30
user_id: 1 | date: 2026-02-13 | meal_type: lunch     | meal_name: salad bowl            | calories: 380 | protein_g: 28 | carbs_g:  22 | fat_g: 12
user_id: 1 | date: 2026-02-13 | meal_type: dinner    | meal_name: noodles               | calories: 720 | protein_g: 18 | carbs_g: 130 | fat_g: 14
user_id: 1 | date: 2026-02-14 | meal_type: lunch     | meal_name: turkey sandwich       | calories: 490 | protein_g: 36 | carbs_g:  44 | fat_g: 14
user_id: 1 | date: 2026-02-14 | meal_type: dinner    | meal_name: beef rice bowl        | calories: 810 | protein_g: 48 | carbs_g:  88 | fat_g: 24
user_id: 2 | date: 2026-02-01 | meal_type: breakfast | meal_name: oatmeal with banana   | calories: 420 | protein_g: 28 | carbs_g:  68 | fat_g:  8
user_id: 2 | date: 2026-02-01 | meal_type: lunch     | meal_name: grilled chicken rice  | calories: 650 | protein_g: 45 | carbs_g:  72 | fat_g: 16
user_id: 2 | date: 2026-02-01 | meal_type: dinner    | meal_name: beef stir fry         | calories: 780 | protein_g: 52 | carbs_g:  60 | fat_g: 28
user_id: 2 | date: 2026-02-02 | meal_type: breakfast | meal_name: eggs and toast        | calories: 390 | protein_g: 24 | carbs_g:  38 | fat_g: 14
user_id: 2 | date: 2026-02-02 | meal_type: lunch     | meal_name: tuna pasta            | calories: 600 | protein_g: 40 | carbs_g:  80 | fat_g: 14
user_id: 2 | date: 2026-02-02 | meal_type: dinner    | meal_name: salmon with quinoa    | calories: 820 | protein_g: 55 | carbs_g:  70 | fat_g: 30
user_id: 3 | date: 2026-02-10 | meal_type: lunch     | meal_name: grilled chicken salad | calories: 410 | protein_g: 36 | carbs_g:  16 | fat_g: 13
user_id: 3 | date: 2026-02-10 | meal_type: dinner    | meal_name: rice with vegetables  | calories: 520 | protein_g: 14 | carbs_g:  88 | fat_g: 10
user_id: 3 | date: 2026-02-11 | meal_type: lunch     | meal_name: quinoa salad          | calories: 390 | protein_g: 14 | carbs_g:  58 | fat_g: 10
user_id: 3 | date: 2026-02-11 | meal_type: dinner    | meal_name: grilled fish          | calories: 480 | protein_g: 48 | carbs_g:   8 | fat_g: 18
user_id: 3 | date: 2026-02-12 | meal_type: lunch     | meal_name: kale salad            | calories: 320 | protein_g: 10 | carbs_g:  30 | fat_g: 11
user_id: 3 | date: 2026-02-12 | meal_type: dinner    | meal_name: pasta                 | calories: 690 | protein_g: 22 | carbs_g: 110 | fat_g: 18
user_id: 4 | date: 2026-01-22 | meal_type: breakfast | meal_name: oatmeal               | calories: 280 | protein_g:  8 | carbs_g:  52 | fat_g:  5
user_id: 4 | date: 2026-01-22 | meal_type: lunch     | meal_name: chicken salad         | calories: 420 | protein_g: 38 | carbs_g:  18 | fat_g: 14
user_id: 4 | date: 2026-01-22 | meal_type: dinner    | meal_name: steak                 | calories: 750 | protein_g: 62 | carbs_g:   4 | fat_g: 42
user_id: 4 | date: 2026-01-29 | meal_type: breakfast | meal_name: eggs and toast        | calories: 340 | protein_g: 18 | carbs_g:  32 | fat_g: 14
user_id: 4 | date: 2026-01-29 | meal_type: lunch     | meal_name: tuna salad            | calories: 370 | protein_g: 32 | carbs_g:  12 | fat_g: 15
user_id: 4 | date: 2026-01-29 | meal_type: dinner    | meal_name: beef bowl             | calories: 810 | protein_g: 48 | carbs_g:  88 | fat_g: 24
user_id: 5 | date: 2026-02-10 | meal_type: breakfast | meal_name: protein shake         | calories: 350 | protein_g: 40 | carbs_g:  28 | fat_g:  8
user_id: 5 | date: 2026-02-10 | meal_type: lunch     | meal_name: grilled salmon        | calories: 620 | protein_g: 55 | carbs_g:  10 | fat_g: 30
user_id: 5 | date: 2026-02-10 | meal_type: dinner    | meal_name: chicken rice bowl     | calories: 720 | protein_g: 50 | carbs_g:  80 | fat_g: 20
user_id: 5 | date: 2026-02-11 | meal_type: breakfast | meal_name: greek yogurt          | calories: 280 | protein_g: 22 | carbs_g:  30 | fat_g:  6
user_id: 5 | date: 2026-02-11 | meal_type: lunch     | meal_name: tuna wrap             | calories: 490 | protein_g: 38 | carbs_g:  44 | fat_g: 12
user_id: 5 | date: 2026-02-11 | meal_type: dinner    | meal_name: beef stir fry         | calories: 780 | protein_g: 52 | carbs_g:  60 | fat_g: 28

-- UserNote --
user_id: 1 | note: I want to lose 5 kg in 2 months, aiming for steady progress
user_id: 2 | note: I work out 3 times per week, sometimes skip workouts, mostly light activity
user_id: 3 | note: I usually skip breakfast and only eat salad for lunch and dinner
user_id: 4 | note: I cannot eat peanuts or shellfish due to allergies
user_id: 5 | note: I recently had knee surgery, cannot exercise, can only lose weight through diet
"""

print("Unified test input ready.")
print(f"Total lines: {len(test_input.strip().splitlines())}")

Unified test input ready.
Total lines: 96


In [ ]:
import random

# -------------------------------------------------------------------------
# Parse test_input into structured dicts for random user selection.
# -------------------------------------------------------------------------

def parse_table(raw, table_name):
    """Extract all rows from a named table block as list of dicts."""
    rows = []
    inside = False
    for line in raw.strip().splitlines():
        line = line.strip()
        if line == f"-- {table_name} --":
            inside = True
            continue
        if inside:
            if line.startswith("--") and line.endswith("--"):
                break
            if not line:
                continue
            row = {}
            for part in line.split(" | "):
                if ": " in part:
                    k, v = part.split(": ", 1)
                    row[k.strip()] = v.strip()
            if row:
                rows.append(row)
    return rows

def filter_user(rows, user_id):
    """Keep only rows for a given user_id."""
    return [r for r in rows if r.get("user_id") == str(user_id)]

def rows_to_text(table_name, rows):
    """Render a list of row dicts back to pipe-separated text."""
    if not rows:
        return f"-- {table_name} --\n(no records)"
    lines = [f"-- {table_name} --"]
    for r in rows:
        lines.append(" | ".join(f"{k}: {v}" for k, v in r.items()))
    return "\n".join(lines)

# Parse all tables once
_profile_rows   = parse_table(test_input, "UserProfile")
_weight_rows    = parse_table(test_input, "WeightLog")
_bodyfat_rows   = parse_table(test_input, "BodyFatLog")
_meal_rows      = parse_table(test_input, "MealLog")
_note_rows      = parse_table(test_input, "UserNote")

ALL_USER_IDS         = [int(r["user_id"]) for r in _profile_rows]
FAT_LOSS_USER_IDS    = [int(r["user_id"]) for r in _profile_rows if r.get("goal") == "fat_loss"]
MUSCLE_GAIN_USER_IDS = [int(r["user_id"]) for r in _profile_rows if r.get("goal") == "muscle_gain"]

print(f"All users       : {ALL_USER_IDS}")
print(f"Fat loss users  : {FAT_LOSS_USER_IDS}")
print(f"Muscle gain users: {MUSCLE_GAIN_USER_IDS}")


def build_full_input(user_id):
    """Return full database text filtered to one user (all tables)."""
    uid = str(user_id)
    return "\n\n".join([
        rows_to_text("UserProfile", filter_user(_profile_rows, uid)),
        rows_to_text("WeightLog",   filter_user(_weight_rows,  uid)),
        rows_to_text("BodyFatLog",  filter_user(_bodyfat_rows, uid)),
        rows_to_text("MealLog",     filter_user(_meal_rows,    uid)),
        rows_to_text("UserNote",    filter_user(_note_rows,    uid)),
    ])

def build_note_only_input(user_id):
    """Return only the UserNote row for one user."""
    return rows_to_text("UserNote", filter_user(_note_rows, str(user_id)))

print("DB parse helpers ready.")


## 3. Helper Functions

In [ ]:
def load_model(model_id):
    """Load tokenizer and model. Falls back to trust_remote_code=True if needed."""
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, device_map="auto", torch_dtype=torch.float16, trust_remote_code=False
        )
    except Exception as e:
        print(f"  [Retrying with trust_remote_code=True] {e}")
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, use_fast=False)
        model = AutoModelForCausalLM.from_pretrained(
            model_id, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True
        )
    return tokenizer, model


def release_model(model=None, tokenizer=None):
    """Free GPU/CPU memory after a model run."""
    if model is not None:
        del model
    if tokenizer is not None:
        del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# -------------------------------------------------------------------------
# User selection rules per task type and prompt index (0-based):
#
#   structured_extraction -> random user from ALL_USER_IDS (UserNote only)
#   text_generation       -> no input (empty string)
#   others                -> random user from ALL_USER_IDS (full input)
# -------------------------------------------------------------------------

def select_user_and_input(task_type, prompt_idx):
    """
    Returns (user_id, input_text) for a given task and prompt position.
    user_id is None for text_generation.
    """
    if task_type == "text_generation":
        return None, ""

    if task_type == "structured_extraction":
        uid = random.choice(ALL_USER_IDS)
        return uid, build_note_only_input(uid)

    if task_type == "prediction" and prompt_idx == 0:
        uid = random.choice(FAT_LOSS_USER_IDS)
    else:
        uid = random.choice(ALL_USER_IDS)

    return uid, build_full_input(uid)


def run_model(model_id, prompts_by_type, all_results, result_file=RESULT_FILE):
    """
    Load one model, run all 25 prompts.
    For each prompt, randomly select a user according to task rules,
    print the selected user_id, and pass the scoped input to the model.
    """
    W = 70
    print("=" * W)
    print(f"  Model : {model_id}")
    print("=" * W)

    new_results = []
    model, tokenizer = None, None

    try:
        print("  Loading model ...")
        tokenizer, model = load_model(model_id)
        print("  Model loaded.\n")

        task_times = {}

        for task_type, prompts in prompts_by_type.items():
            task_total = 0.0

            print(f"  Task: {task_type.upper().replace('_', ' ')}")
            print("  " + "-" * 60)

            for idx, prompt in enumerate(prompts, 1):
                user_id, input_data = select_user_and_input(task_type, idx - 1)

                if user_id is not None:
                    print(f"  [{idx}/5] Selected user_id : {user_id}")
                else:
                    print(f"  [{idx}/5] No user input (text_generation)")

                if input_data:
                    formatted = (
                        f"Instruction:\n{prompt}\n\n"
                        f"User Data:\n{input_data}\n\n"
                        f"Answer:"
                    )
                else:
                    formatted = f"Instruction:\n{prompt}\n\nAnswer:"

                tokenized = tokenizer(formatted, return_tensors="pt").to(model.device)

                t0 = time.time()
                outputs = model.generate(**tokenized, max_new_tokens=100, do_sample=False)
                elapsed = round(time.time() - t0, 2)
                task_total += elapsed

                response = tokenizer.decode(
                    outputs[0][tokenized.input_ids.shape[-1]:],
                    skip_special_tokens=True
                ).strip()

                print(f"         Prompt   : {prompt[:90]}{'...' if len(prompt) > 90 else ''}")
                print(f"         Response : {response[:200]}{'...' if len(response) > 200 else ''}")
                print(f"         Time     : {elapsed}s")
                print()

                new_results.append({
                    "model":      model_id,
                    "type":       task_type,
                    "user_id":    user_id,
                    "prompt":     prompt,
                    "response":   response,
                    "time_taken": elapsed,
                })

            task_times[task_type] = round(task_total, 2)
            print(f"  [{task_type}] completed  (total: {task_times[task_type]}s)\n")

        total_time = sum(task_times.values())
        print("  " + "=" * 50)
        print(f"  {'Task':<28} {'Time (s)':>10}")
        print("  " + "-" * 40)
        for t, s in task_times.items():
            print(f"  {t:<28} {s:>10.2f}")
        print("  " + "-" * 40)
        print(f"  {'Total':<28} {total_time:>10.2f}")
        print("  " + "=" * 50)

    except Exception as e:
        print(f"  [ERROR] Failed to run {model_id}: {e}")

    finally:
        release_model(model, tokenizer)
        model, tokenizer = None, None

    all_results.extend(new_results)
    with open(result_file, "w", encoding="utf-8") as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)
    print(f"\n  Saved to {result_file}  ({len(all_results)} total records)\n")

    return new_results


print("Helper functions defined.")


## 4. Model Evaluation

Run each cell independently. Results are saved to `llm_test_results.json` after every model.

### Ultra-Light (< 1B parameters)

#### `Qwen/Qwen2.5-0.5B-Instruct`

In [ ]:
# Qwen2.5-0.5B-Instruct
results_Qwen2_5_0_5B_Instruct = run_model(
    "Qwen/Qwen2.5-0.5B-Instruct",
    test_prompts_by_type,
    all_results,
)


#### `HuggingFaceTB/SmolLM2-360M-Instruct`

In [ ]:
# SmolLM2-360M-Instruct
results_SmolLM2_360M_Instruct = run_model(
    "HuggingFaceTB/SmolLM2-360M-Instruct",
    test_prompts_by_type,
    all_results,
)


#### `EleutherAI/pythia-410m`

In [ ]:
# pythia-410m
results_pythia_410m = run_model(
    "EleutherAI/pythia-410m",
    test_prompts_by_type,
    all_results,
)


### Small (1B - 3B parameters)

#### `TinyLlama/TinyLlama-1.1B-Chat-v1.0`

In [ ]:
# TinyLlama-1.1B-Chat-v1.0
results_TinyLlama_1_1B_Chat_v1_0 = run_model(
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    test_prompts_by_type,
    all_results,
)


#### `stabilityai/stablelm-2-zephyr-1_6b`

In [ ]:
# stablelm-2-zephyr-1_6b
results_stablelm_2_zephyr_1_6b = run_model(
    "stabilityai/stablelm-2-zephyr-1_6b",
    test_prompts_by_type,
    all_results,
)


#### `microsoft/phi-2`

In [ ]:
# phi-2
results_phi_2 = run_model(
    "microsoft/phi-2",
    test_prompts_by_type,
    all_results,
)


### Medium (3B - 5B parameters)

#### `Qwen/Qwen2.5-3B-Instruct`

In [ ]:
# Qwen2.5-3B-Instruct
results_Qwen2_5_3B_Instruct = run_model(
    "Qwen/Qwen2.5-3B-Instruct",
    test_prompts_by_type,
    all_results,
)


#### `microsoft/Phi-3-mini-4k-instruct`

In [ ]:
# Phi-3-mini-4k-instruct
results_Phi_3_mini_4k_instruct = run_model(
    "microsoft/Phi-3-mini-4k-instruct",
    test_prompts_by_type,
    all_results,
)


#### `stabilityai/stablelm-zephyr-3b`

In [ ]:
# stablelm-zephyr-3b
results_stablelm_zephyr_3b = run_model(
    "stabilityai/stablelm-zephyr-3b",
    test_prompts_by_type,
    all_results,
)


### Large (5B - 7B parameters)

#### `HuggingFaceH4/zephyr-7b-beta`

In [ ]:
# zephyr-7b-beta
results_zephyr_7b_beta = run_model(
    "HuggingFaceH4/zephyr-7b-beta",
    test_prompts_by_type,
    all_results,
)


#### `NousResearch/Nous-Hermes-2-Mistral-7B-DPO`

In [ ]:
# Nous-Hermes-2-Mistral-7B-DPO
results_Nous_Hermes_2_Mistral_7B_DPO = run_model(
    "NousResearch/Nous-Hermes-2-Mistral-7B-DPO",
    test_prompts_by_type,
    all_results,
)


#### `tiiuae/falcon-7b-instruct`

In [ ]:
# falcon-7b-instruct
results_falcon_7b_instruct = run_model(
    "tiiuae/falcon-7b-instruct",
    test_prompts_by_type,
    all_results,
)


### Extra-Large (7B - 10B+ parameters)

#### `01-ai/Yi-1.5-9B-Chat`

In [ ]:
# Yi-1.5-9B-Chat
results_Yi_1_5_9B_Chat = run_model(
    "01-ai/Yi-1.5-9B-Chat",
    test_prompts_by_type,
    all_results,
)


#### `upstage/SOLAR-10.7B-Instruct-v1.0`

In [ ]:
# SOLAR-10.7B-Instruct-v1.0
results_SOLAR_10_7B_Instruct_v1_0 = run_model(
    "upstage/SOLAR-10.7B-Instruct-v1.0",
    test_prompts_by_type,
    all_results,
)


#### `lmsys/vicuna-7b-v1.5`

In [ ]:
# vicuna-7b-v1.5
results_vicuna_7b_v1_5 = run_model(
    "lmsys/vicuna-7b-v1.5",
    test_prompts_by_type,
    all_results,
)


## 5. Full Results Summary

In [ ]:
import pandas as pd

df = pd.DataFrame(all_results)

print("=" * 70)
print("PER-MODEL TIMING SUMMARY")
print("=" * 70)
summary = (
    df.groupby("model")["time_taken"]
    .agg(tasks="count", total_s="sum", avg_s="mean")
    .reset_index()
    .sort_values("total_s")
)
summary["total_s"] = summary["total_s"].round(2)
summary["avg_s"]   = summary["avg_s"].round(2)
print(summary.to_string(index=False))

print("\n" + "=" * 70)
print("AVERAGE RESPONSE TIME BY TASK TYPE")
print("=" * 70)
task_summary = df.groupby("type")["time_taken"].mean().round(2).reset_index()
task_summary.columns = ["task_type", "avg_time_s"]
print(task_summary.to_string(index=False))

print("\n" + "=" * 70)
print(f"FULL RESULTS  ({len(df)} rows)")
print("=" * 70)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.max_rows", 400)
display(df[["model", "type", "user_id", "prompt", "response", "time_taken"]])
